# 01 Exploratory data analysis: credit card customer segmentation

**Business question:** Which groups of credit card customers exist based on how they spend, pay, and borrow, and what does each group look like in business terms?

**Data:** the CC GENERAL dataset. It has about 8,950 rows (one per credit card holder) and 18 columns, with behaviour measured over the last 6 months. We found it as `CC_GENERAL.csv` and copied it to `data/raw/credit_card.csv`.

**How to read this notebook:** every code cell has a short note above it with three lines (what it does, what to look for, what the output means). Imports sit inside the cells that first use them, with a comment saying why.

## Step 1: Early EDA with ydata-profiling

*EDA* means exploratory data analysis: looking at the data before modelling so nothing surprises us later.

**What this cell does:** Finds the project folder and loads the CSV into a table (a pandas DataFrame).  
**What to look for:** That the shape matches what we expect: about 8,950 rows and 18 columns.  
**What the output means:** `shape` is (rows, columns). If it were far off, we would have loaded the wrong file.

In [1]:
# Imported here so this step only loads the library when it runs
from pathlib import Path
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Walk up from the current folder until we see pyproject.toml (nbconvert runs inside notebooks/)."""
    return next(folder for folder in [start, *start.parents] if (folder / "pyproject.toml").exists())


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "raw" / "credit_card.csv"
REPORTS_DIR = ROOT / "reports"

df = pd.read_csv(DATA_PATH)  # read the CSV into a DataFrame
print(f"{df.shape=}")
df.head()  # first 5 rows, to eyeball the columns

df.shape=(8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


**What this cell does:** Lists each column's data type (dtype) and counts missing values and duplicate rows.  
**What to look for:** Any column that is not numeric (we expect only CUST_ID), the columns with missing values, and the duplicate count.  
**What the output means:** Numeric dtypes mean we can compute on them directly. Missing values must be handled before clustering because KMeans cannot accept blanks. Duplicated rows would count one customer twice.

In [2]:
print(f"{df.dtypes.value_counts().to_dict()=}")  # how many columns of each type
print(f"{df.duplicated().sum()=}")  # fully identical rows
print(f"{df['CUST_ID'].is_unique=}")  # every customer ID should appear exactly once

# One-liner: count blanks per column, keep only columns that have some, show count and percent.
# isna() marks blanks as True, sum() counts them, and the mask [> 0] drops complete columns.
missing = df.isna().sum()
missing_table = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})[missing > 0]
missing_table

df.dtypes.value_counts().to_dict()={dtype('float64'): 14, dtype('int64'): 3, dtype('O'): 1}
df.duplicated().sum()=np.int64(0)
df['CUST_ID'].is_unique=True


,missing,percent
CREDIT_LIMIT,1,0.01
MINIMUM_PAYMENTS,313,3.50


**What this cell does:** Prints summary statistics (count, mean, spread, min, quartiles, max) for the numeric columns.  
**What to look for:** Big gaps between the median (50%) and the max, and columns that are mostly zero.  
**What the output means:** When the max is far above the 75% value, a few customers have extreme values (a long right tail). That matters later because KMeans is pulled by extremes.

In [3]:
numeric_columns = df.drop(columns="CUST_ID").columns.tolist()  # every column except the ID
df[numeric_columns].describe().T.round(2)  # .T flips the table so each feature is one row

,count,mean,std,min,25%,50%,75%,max
BALANCE,8950.0,1564.47,2081.53,0.00,128.28,873.39,2054.14,19043.14
BALANCE_FREQUENCY,8950.0,0.88,0.24,0.00,0.89,1.00,1.00,1.00
PURCHASES,8950.0,1003.20,2136.63,0.00,39.64,361.28,1110.13,49039.57
ONEOFF_PURCHASES,8950.0,592.44,1659.89,0.00,0.00,38.00,577.40,40761.25
INSTALLMENTS_PURCHASES,8950.0,411.07,904.34,0.00,0.00,89.00,468.64,22500.00
CASH_ADVANCE,8950.0,978.87,2097.16,0.00,0.00,0.00,1113.82,47137.21
PURCHASES_FREQUENCY,8950.0,0.49,0.40,0.00,0.08,0.50,0.92,1.00
ONEOFF_PURCHASES_FREQUENCY,8950.0,0.20,0.30,0.00,0.00,0.08,0.30,1.00
PURCHASES_INSTALLMENTS_FREQUENCY,8950.0,0.36,0.40,0.00,0.00,0.17,0.75,1.00
CASH_ADVANCE_FREQUENCY,8950.0,0.14,0.20,0.00,0.00,0.00,0.22,1.50


**What this cell does:** Builds the full ydata-profiling HTML report and saves it in reports/.  
**What to look for:** Open reports/credit_card_profile.html in a browser. Look at the Alerts tab, the Correlations tab, and each variable's histogram.  
**What the output means:** The report automatically flags skew, many zeros, high correlation and missing values. It is a fast first look, not a final answer.

In [4]:
def build_profile(frame: pd.DataFrame, output_path: Path):
    """Create the profiling report and save it as HTML."""
    # Imported here so the notebook opens fast and this step only loads the library when it runs
    from ydata_profiling import ProfileReport

    report = ProfileReport(frame, title="Credit card customers: profile", minimal=False, progress_bar=False)
    report.to_file(output_path)  # writes the interactive HTML page
    return report


REPORTS_DIR.mkdir(exist_ok=True)
profile = build_profile(df.drop(columns="CUST_ID"), REPORTS_DIR / "credit_card_profile.html")  # ID is not a feature
print(f"{(REPORTS_DIR / 'credit_card_profile.html').exists()=}")

/tmp/ipykernel_1038/232935934.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


  0%|          | 0/17 [00:00<?, ?it/s]

 29%|██▉       | 5/17 [00:00<00:00, 32.54it/s]

 59%|█████▉    | 10/17 [00:00<00:00, 31.57it/s]

100%|██████████| 17/17 [00:00<00:00, 44.62it/s]

(REPORTS_DIR / 'credit_card_profile.html').exists()=True


**What this cell does:** Prints the warnings (alerts) that ydata-profiling raised, so we can quote them in our findings.  
**What to look for:** Alert types such as SKEWED, ZEROS, HIGH_CORRELATION, MISSING.  
**What the output means:** Each alert is a data-quality hint. We read them here so our written findings come from real output, not guesses.

In [5]:
alerts = profile.get_description().alerts  # list of Alert objects
alert_table = pd.DataFrame({"column": [a.column_name for a in alerts], "alert": [a.alert_type.name for a in alerts]})
print(alert_table.groupby("alert")["column"].apply(lambda cols: ", ".join(sorted(cols))).to_string())

alert
HIGH_CORRELATION    BALANCE, BALANCE_FREQUENCY, CASH_ADVANCE, CASH...
MISSING                                              MINIMUM_PAYMENTS
ZEROS               CASH_ADVANCE, CASH_ADVANCE_FREQUENCY, CASH_ADV...


**What this cell does:** Checks three structural facts the report may hint at: whether PURCHASES equals one-off plus installment purchases, how many rows have zero purchases, and how many have MINIMUM_PAYMENTS above PAYMENTS.  
**What to look for:** The share of rows where the purchase identity holds, and the counts of zero-purchase and odd payment rows.  
**What the output means:** If PURCHASES is just the sum of two other columns, it adds no new information (redundancy). Customers with zero purchases are a natural group (cash-advance or inactive users).

In [6]:
# Does PURCHASES equal ONEOFF + INSTALLMENTS? np.isclose allows tiny rounding differences.
import numpy as np  # imported here because this is the first cell that needs numpy

identity_holds = np.isclose(df["PURCHASES"], df["ONEOFF_PURCHASES"] + df["INSTALLMENTS_PURCHASES"], atol=0.01)
print(f"{identity_holds.mean()=:.4f}")
print(f"{(df['PURCHASES'] == 0).sum()=}")
print(f"{(df['MINIMUM_PAYMENTS'] > df['PAYMENTS']).sum()=}")
print(f"{df['TENURE'].value_counts().sort_index().to_dict()=}")
print(f"{df['CREDIT_LIMIT'].isna().sum()=}, {df['MINIMUM_PAYMENTS'].isna().sum()=}")

identity_holds.mean()=0.9979
(df['PURCHASES'] == 0).sum()=np.int64(2044)
(df['MINIMUM_PAYMENTS'] > df['PAYMENTS']).sum()=np.int64(2365)
df['TENURE'].value_counts().sort_index().to_dict()={6: 204, 7: 190, 8: 196, 9: 175, 10: 236, 11: 365, 12: 7584}
df['CREDIT_LIMIT'].isna().sum()=np.int64(1), df['MINIMUM_PAYMENTS'].isna().sum()=np.int64(313)


### Five findings from the profile

- **The table is clean in structure.** There are 8,950 rows, no duplicate rows, and every `CUST_ID` is unique. *Why it matters:* one row is one customer, so no de-duplication is needed and cluster sizes can be read directly as shares of customers.
- **Missing values are few and limited to two columns.** `MINIMUM_PAYMENTS` is missing for 313 customers (3.50%) and `CREDIT_LIMIT` for just 1. *Why it matters:* KMeans cannot accept blanks, but this is small enough to fill in with a simple rule inside a pipeline (Step 4) instead of throwing customers away.
- **Money columns are extremely right-skewed.** Skewness (how lopsided a distribution is; 0 is symmetric) is 13.6 for `MINIMUM_PAYMENTS`, 10.0 for `ONEOFF_PURCHASES` and 8.1 for `PURCHASES`. The top `PURCHASES` value is about 49,000 while 99% of customers are below about 9,000. *Why it matters:* KMeans measures straight-line distance, so a few huge values would drag the centres and hide the smaller groups. We will transform the skewed columns (Step 5).
- **Many columns are mostly zeros.** 2,044 customers (about 23%) have made no purchases at all, and the cash advance columns are about 52% zeros. *Why it matters:* these zeros are real behaviour, not errors. They hint at natural groups (for example customers who only take cash advances), and they mean we should use `log1p` (log of 1 + x), which is safe at zero, rather than a plain log.
- **Several features say nearly the same thing.** `PURCHASES` equals `ONEOFF_PURCHASES` plus `INSTALLMENTS_PURCHASES` in 99.8% of rows, and pairs such as `CASH_ADVANCE_FREQUENCY` and `CASH_ADVANCE_TRX` move together almost perfectly (Spearman correlation 0.98). *Why it matters:* duplicated information counts twice in a distance calculation, so we will decide which features to drop in Step 3.

**Other things noticed:** 2,365 customers have a minimum payment larger than their payments (we will look at this in Step 4), and 7,584 of 8,950 customers (85%) have `TENURE` of 12 months, so tenure carries little information to separate groups.

---
## Step 2: Univariate EDA and normality testing

*Univariate* means looking at one column at a time. For each numeric feature we draw three plots:

- **Histogram with KDE:** bars show how many customers fall in each value range. The KDE (kernel density estimate) is a smooth line drawn over the bars.
- **Boxplot:** the box covers the middle 50% of customers, the line inside is the median, and dots beyond the whiskers are unusually large or small values.
- **Q-Q plot** (quantile-quantile): compares our data with a perfect bell curve. If the dots sit on the red line, the data is roughly normal. Curving away means skew or heavy tails.

**What this cell does:** Computes skewness and kurtosis for every numeric feature and sorts them from most skewed to least.  
**What to look for:** Skewness far from 0 (above +1 is strongly right-skewed) and kurtosis far from 0.  
**What the output means:** Skewness measures lopsidedness: positive means a long tail to the right. Kurtosis here is *excess* kurtosis (0 for a normal curve): large positive values mean heavy tails, so more extreme values than a bell curve would give.

In [7]:
from scipy import stats  # imported here because this is the first cell that needs scipy

features = df.drop(columns="CUST_ID").columns.tolist()  # the 17 numeric features

shape_table = pd.DataFrame({
    "skewness": [stats.skew(df[col].dropna()) for col in features],      # lopsidedness of each column
    "excess_kurtosis": [stats.kurtosis(df[col].dropna()) for col in features],  # tail heaviness (0 = normal)
}, index=features).round(2).sort_values("skewness", ascending=False)
shape_table

,skewness,excess_kurtosis
MINIMUM_PAYMENTS,13.62,283.82
ONEOFF_PURCHASES,10.04,164.10
PURCHASES,8.14,111.33
INSTALLMENTS_PURCHASES,7.30,96.52
PAYMENTS,5.91,54.74
CASH_ADVANCE_TRX,5.72,61.61
CASH_ADVANCE,5.17,52.87
PURCHASES_TRX,4.63,34.77
BALANCE,2.39,7.67
PRC_FULL_PAYMENT,1.94,2.43


**What this cell does:** Draws the histogram+KDE, boxplot and Q-Q plot for every feature, in three batches, and saves the figures in reports/figures/.  
**What to look for:** Histograms piled up at the left with a long tail, boxplots with many dots on the right, and Q-Q dots that bend away from the red line.  
**What the output means:** Together they show the same story as the skewness table, but visually. Bar piles against zero show the many-zeros problem from Step 1.

In [8]:
import matplotlib.pyplot as plt  # imported here because plotting starts in this cell
import seaborn as sns            # nicer statistical plots on top of matplotlib

FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def plot_univariate(feature_names: list[str], file_name: str) -> None:
    """One row per feature: histogram+KDE, boxplot, Q-Q plot."""
    fig, axes = plt.subplots(len(feature_names), 3, figsize=(14, 2.6 * len(feature_names)))
    for row, name in enumerate(feature_names):
        values = df[name].dropna()
        sns.histplot(values, kde=True, ax=axes[row, 0], color="steelblue")  # bars + smooth KDE line
        axes[row, 0].set_title(f"{name}: histogram + KDE", fontsize=9)
        sns.boxplot(x=values, ax=axes[row, 1], color="lightgray")           # box = middle 50%, dots = outliers
        axes[row, 1].set_title(f"{name}: boxplot", fontsize=9)
        stats.probplot(values, dist="norm", plot=axes[row, 2])              # Q-Q plot against a normal curve
        axes[row, 2].set_title(f"{name}: Q-Q plot", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / file_name, dpi=110)
    plt.show()


# Three batches of features (6, 6, 5) so each figure stays readable
for batch_number, start in enumerate(range(0, len(features), 6), start=1):
    plot_univariate(features[start:start + 6], f"univariate_{batch_number}.png")

### Normality tests

A *normality test* asks: "could this column plausibly have come from a normal (bell curve) distribution?" Each test gives a **p-value**. If p is below 0.05 (our alpha, the cut-off for "surprising"), we reject normality.

- **Shapiro-Wilk** is a strong test, but it becomes overly strict on large samples: with thousands of rows it flags even tiny, harmless differences from a bell curve. That is why we run it on a random sample of at most 5,000 rows.
- **D'Agostino K-squared** (`scipy.stats.normaltest`) combines skewness and kurtosis into one test. We run it on the full data.

**What this cell does:** Runs both normality tests on every feature and puts the results in one table with a yes/no column.  
**What to look for:** The `normal_at_0.05` column, and how small the p-values are.  
**What the output means:** p-values below 0.05 mean the test rejects normality. We call a feature normal only if it passes both tests. A p-value printed as 0.00e+00 just means it is smaller than the computer can store. (SciPy warns that Shapiro p-values are approximate above 5,000 rows, which is one more reason we sample.)

In [9]:
rows = []
for name in features:
    values = df[name].dropna()
    sample = values.sample(n=min(5000, len(values)), random_state=42)  # fixed seed so results repeat
    shapiro_p = stats.shapiro(sample).pvalue          # Shapiro-Wilk on the sample
    dagostino_p = stats.normaltest(values).pvalue     # D'Agostino K-squared on all rows
    rows.append({
        "feature": name,
        "shapiro_p (sample)": shapiro_p,
        "dagostino_p (full)": dagostino_p,
        # one-liner: "yes" only if BOTH p-values are above alpha (0.05), otherwise "no"
        "normal_at_0.05": "yes" if min(shapiro_p, dagostino_p) > 0.05 else "no",
    })

normality_table = pd.DataFrame(rows).set_index("feature")
normality_table.style.format({"shapiro_p (sample)": "{:.2e}", "dagostino_p (full)": "{:.2e}"})  # sci notation for tiny p

,shapiro_p (sample),dagostino_p (full),normal_at_0.05
feature,,,
BALANCE,9.39e-68,0.00e+00,no
BALANCE_FREQUENCY,9.18e-76,0.00e+00,no
PURCHASES,2.87e-82,0.00e+00,no
ONEOFF_PURCHASES,4.35e-86,0.00e+00,no
INSTALLMENTS_PURCHASES,8.93e-82,0.00e+00,no
CASH_ADVANCE,7.10e-79,0.00e+00,no
PURCHASES_FREQUENCY,2.18e-57,0.00e+00,no
ONEOFF_PURCHASES_FREQUENCY,9.83e-69,0.00e+00,no
PURCHASES_INSTALLMENTS_FREQUENCY,2.82e-62,0.00e+00,no


**What this cell does:** Counts how many features are strongly right-skewed and lists the ones that are not.  
**What to look for:** The number of features with skewness above 1, and which features are the exceptions.  
**What the output means:** This confirms or corrects our expectation that credit card amounts are right-skewed, and it tells us which columns need a transform in Step 5.

In [10]:
strongly_right_skewed = shape_table[shape_table["skewness"] > 1].index.tolist()
others = shape_table[shape_table["skewness"] <= 1]
print(f"{len(strongly_right_skewed)=} of {len(features)}")
print(f"{strongly_right_skewed=}")
print("Not strongly right-skewed:")
print(others.to_string())
print(f"{(normality_table['normal_at_0.05'] == 'yes').sum()=} features pass both normality tests")

len(strongly_right_skewed)=13 of 17
strongly_right_skewed=['MINIMUM_PAYMENTS', 'ONEOFF_PURCHASES', 'PURCHASES', 'INSTALLMENTS_PURCHASES', 'PAYMENTS', 'CASH_ADVANCE_TRX', 'CASH_ADVANCE', 'PURCHASES_TRX', 'BALANCE', 'PRC_FULL_PAYMENT', 'CASH_ADVANCE_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'CREDIT_LIMIT']
Not strongly right-skewed:
                                  skewness  excess_kurtosis
PURCHASES_INSTALLMENTS_FREQUENCY      0.51            -1.40
PURCHASES_FREQUENCY                   0.06            -1.64
BALANCE_FREQUENCY                    -2.02             3.09
TENURE                               -2.94             7.69
(normality_table['normal_at_0.05'] == 'yes').sum()=np.int64(0) features pass both normality tests


### What Step 2 tells us

- **Our expectation was mostly right, with an important exception.** 13 of 17 features have skewness above 1. All eight money amount columns and the count columns are strongly right-skewed (skewness from 2.4 for `BALANCE` up to 13.6 for `MINIMUM_PAYMENTS`), with huge excess kurtosis (up to 284), meaning a few very large customers stretch the tail.
- **The exceptions are the bounded columns.** `PURCHASES_FREQUENCY` (skew 0.06) and `PURCHASES_INSTALLMENTS_FREQUENCY` (0.51) are flat, with piles at 0 and 1, so they are not right-skewed. `BALANCE_FREQUENCY` (-2.0) and `TENURE` (-2.9) are *left*-skewed because most customers sit at the maximum (1.0 and 12 months). A log transform would be the wrong tool for these.
- **No feature is normal.** Zero of 17 pass both tests. Shapiro-Wilk p-values (on 5,000 sampled rows) are all below 1e-55 and the D'Agostino p-values underflow to 0. Because the skewness and kurtosis are so large, this verdict does not depend on Shapiro being too strict: even a lenient test would reject. Even `PURCHASES_FREQUENCY`, with skewness near 0, fails because its shape is flat, not bell-shaped.
- **What this means for KMeans.** KMeans does not need normal data. The real problem is scale and extremes: it uses straight-line distance, so a feature in the tens of thousands of dollars would swamp one measured from 0 to 1, and a few giant customers would pull the cluster centres. So we will log-transform the skewed *amount and count* columns (Step 5), leave the bounded 0 to 1 columns alone, and then put everything on one scale.

---
## Step 3: Correlation and multicollinearity

*Correlation* measures how strongly two columns move together (from -1 to +1). *Multicollinearity* means several features carry nearly the same information.

We use **Spearman** correlation instead of Pearson. Pearson measures straight-line relationships and is thrown off by extreme values; Spearman compares only the *ranks* (who is first, second, third), so skew and outliers hardly affect it.

**What this cell does:** Computes the Spearman correlation between every pair of features and draws an annotated heatmap.  
**What to look for:** Dark red blocks (strong positive) and dark blue (strong negative). Look for groups of features that form a block.  
**What the output means:** Each cell shows how closely two features rank customers the same way. Values near +1 mean one feature adds almost no new information beyond the other.

In [11]:
spearman = df[features].corr(method="spearman")  # rank-based correlation, robust to skew

plt.figure(figsize=(15, 12))
sns.heatmap(spearman, annot=True, fmt=".2f", annot_kws={"size": 7}, cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Spearman correlation between features")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "spearman_heatmap.png", dpi=110)
plt.show()

**What this cell does:** Lists every pair of different features with an absolute Spearman correlation above 0.8.  
**What to look for:** Which pairs appear and how many times each feature shows up.  
**What the output means:** These are the redundant pairs. A feature that appears in many pairs is a good candidate to drop or to keep as the single representative of its group.

In [12]:
import numpy as np  # needed to keep only the upper triangle so each pair is listed once

upper_triangle = spearman.where(np.triu(np.ones(spearman.shape, dtype=bool), k=1))  # k=1 skips the diagonal
high_pairs = (
    upper_triangle.stack()                                  # turn the matrix into (feature_a, feature_b) -> value
    .rename_axis(["feature_a", "feature_b"])               # name the two index levels
    .reset_index(name="spearman")                           # turn them into columns
    .loc[lambda pairs: pairs["spearman"].abs() > 0.8]       # keep strong pairs only
    .sort_values("spearman", key=abs, ascending=False)
    .round(2)
)
display(high_pairs.reset_index(drop=True))
print("How often each feature appears in a high pair:")
print(pd.concat([high_pairs["feature_a"], high_pairs["feature_b"]]).value_counts().to_string())

,feature_a,feature_b,spearman
0,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,0.98
1,ONEOFF_PURCHASES,ONEOFF_PURCHASES_FREQUENCY,0.95
2,CASH_ADVANCE,CASH_ADVANCE_TRX,0.95
3,CASH_ADVANCE,CASH_ADVANCE_FREQUENCY,0.94
4,PURCHASES_FREQUENCY,PURCHASES_TRX,0.92
5,INSTALLMENTS_PURCHASES,PURCHASES_INSTALLMENTS_FREQUENCY,0.92
6,BALANCE,MINIMUM_PAYMENTS,0.90
7,PURCHASES,PURCHASES_TRX,0.89
8,PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,0.85


How often each feature appears in a high pair:
CASH_ADVANCE_FREQUENCY              2
CASH_ADVANCE                        2
PURCHASES_FREQUENCY                 2
CASH_ADVANCE_TRX                    2
PURCHASES_INSTALLMENTS_FREQUENCY    2
PURCHASES_TRX                       2
ONEOFF_PURCHASES                    1
INSTALLMENTS_PURCHASES              1
PURCHASES                           1
BALANCE                             1
ONEOFF_PURCHASES_FREQUENCY          1
MINIMUM_PAYMENTS                    1


### Decision: which features to keep

Nine pairs are above 0.8. They fall into clear families. Our rule: when two features are the *same quantity counted differently* (an amount versus a count or a frequency), keep one and drop the others, so one behaviour does not get counted two or three times in the distance.

| Pair (Spearman) | Decision | Reason |
|---|---|---|
| `CASH_ADVANCE_FREQUENCY` and `CASH_ADVANCE_TRX` (0.98), `CASH_ADVANCE` with each (0.94, 0.95) | **Keep `CASH_ADVANCE`, drop the other two** | Three versions of "how much cash does this customer borrow". The amount is the one with direct business meaning (money borrowed at a high rate). |
| `PURCHASES` and `PURCHASES_TRX` (0.89), `PURCHASES_FREQUENCY` and `PURCHASES_TRX` (0.92) | **Drop `PURCHASES` and `PURCHASES_TRX`, keep `PURCHASES_FREQUENCY`** | `PURCHASES` is (almost exactly) one-off plus installments, which we keep separately, so it adds no new information. The transaction count says what the frequency already says. |
| `ONEOFF_PURCHASES` and `ONEOFF_PURCHASES_FREQUENCY` (0.95) | **Keep the amount, drop the frequency** | Same behaviour, and overall purchase frequency is kept through `PURCHASES_FREQUENCY`. |
| `INSTALLMENTS_PURCHASES` and `PURCHASES_INSTALLMENTS_FREQUENCY` (0.92), `PURCHASES_FREQUENCY` and `PURCHASES_INSTALLMENTS_FREQUENCY` (0.85) | **Keep the amount, drop the frequency** | Same reasoning, and this also resolves the 0.85 pair. |
| `BALANCE` and `MINIMUM_PAYMENTS` (0.90) | **Keep both** | This is the one exception. They are different things (what is owed versus the smallest payment due), 0.90 is close to our line, and the minimum payment helps show who under-pays. We will check in notebook 03 that dropping it does not change the clusters. |

**Result:** we drop 6 features (`PURCHASES`, `PURCHASES_TRX`, `CASH_ADVANCE_TRX`, `CASH_ADVANCE_FREQUENCY`, `ONEOFF_PURCHASES_FREQUENCY`, `PURCHASES_INSTALLMENTS_FREQUENCY`) and keep 11 for clustering. In Step 8 we still show the dropped columns when we describe each cluster, because they are useful for explaining it in business terms.

**What this cell does:** Records the six dropped features in one place and checks which strong pairs remain among the 11 kept features.  
**What to look for:** Only the BALANCE and MINIMUM_PAYMENTS pair should remain above 0.8.  
**What the output means:** This proves the drop list does what we claimed, instead of trusting the table above by eye.

In [13]:
FEATURES_TO_DROP = [
    "PURCHASES", "PURCHASES_TRX", "CASH_ADVANCE_TRX",
    "CASH_ADVANCE_FREQUENCY", "ONEOFF_PURCHASES_FREQUENCY", "PURCHASES_INSTALLMENTS_FREQUENCY",
]
kept_features = [name for name in features if name not in FEATURES_TO_DROP]
print(f"{len(kept_features)=}")

kept_pairs = high_pairs[high_pairs["feature_a"].isin(kept_features) & high_pairs["feature_b"].isin(kept_features)]
print("Strong pairs left among kept features:")
print(kept_pairs.to_string(index=False))

len(kept_features)=11
Strong pairs left among kept features:
feature_a        feature_b  spearman
  BALANCE MINIMUM_PAYMENTS       0.9


---
## Step 4: Missing values and outliers

### Missing values: the options

1. **Drop the rows.** Simple, but we lose customers (and the customers who are missing may be a special group, which would bias the result).
2. **Median imputation.** Fill each blank with the column's median (the middle value). Robust to extreme values, which matters because these columns are heavily skewed. Fast and easy to explain.
3. **Model-based imputation** (for example KNN or iterative imputation). Predicts each blank from the other columns. More flexible, but slower, harder to explain, and not worth it when only 3.5% is missing in one column.

First we check *why* the values are missing, because the reason affects the choice.

**What this cell does:** Looks at the customers whose MINIMUM_PAYMENTS or CREDIT_LIMIT is missing, and compares their other columns with everyone else.  
**What to look for:** Whether the missing rows share a pattern, for example all having PAYMENTS equal to 0.  
**What the output means:** If blanks follow a pattern, they are not random. That tells us whether a plain median fill is a fair guess.

In [14]:
missing_min_payment = df["MINIMUM_PAYMENTS"].isna()
print(f"{missing_min_payment.sum()=}")
print("PAYMENTS for customers missing MINIMUM_PAYMENTS:")
print(df.loc[missing_min_payment, "PAYMENTS"].describe().round(2).to_string())
print(f"{(df.loc[missing_min_payment, 'PAYMENTS'] == 0).mean()=:.3f}")
print(f"{(df.loc[~missing_min_payment, 'PAYMENTS'] == 0).sum()=}  (zero payments among rows that have a minimum payment)")
print("Row missing CREDIT_LIMIT:")
df[df["CREDIT_LIMIT"].isna()].T

missing_min_payment.sum()=np.int64(313)
PAYMENTS for customers missing MINIMUM_PAYMENTS:
count      313.00
mean       322.29
std       1996.66
min          0.00
25%          0.00
50%          0.00
75%          0.00
max      29272.49
(df.loc[missing_min_payment, 'PAYMENTS'] == 0).mean()=0.767
(df.loc[~missing_min_payment, 'PAYMENTS'] == 0).sum()=np.int64(0)  (zero payments among rows that have a minimum payment)
Row missing CREDIT_LIMIT:


,5203
CUST_ID,C15349
BALANCE,18.400472
BALANCE_FREQUENCY,0.166667
PURCHASES,0.0
ONEOFF_PURCHASES,0.0
INSTALLMENTS_PURCHASES,0.0
CASH_ADVANCE,186.853063
PURCHASES_FREQUENCY,0.0
ONEOFF_PURCHASES_FREQUENCY,0.0
PURCHASES_INSTALLMENTS_FREQUENCY,0.0


### What the missing values tell us

- 313 customers lack `MINIMUM_PAYMENTS`, and **77% of them (240) made exactly zero payments**. Among customers who do have a minimum payment, none have zero payments. So the blank is not random: it most likely means "no payment was due or made". (We did not look for a pattern in the remaining 73.)
- `CREDIT_LIMIT` is missing for a single customer, with tiny balance and a tenure of 6 months. Nothing special.

### Decision: median imputation, inside a Pipeline

- **Why not drop rows?** We would lose 313 customers who are probably a real group (inactive on payments), and the clusters would not describe them.
- **Why not model-based?** For 3.5% of one column, KNN or iterative imputation adds complexity and run time with little gain, and is harder to explain to a business reader.
- **Why median?** Both columns are heavily skewed, so the mean (pulled up by giants) would be a poor guess. The median (about 312 for `MINIMUM_PAYMENTS`, 3,000 for `CREDIT_LIMIT`) is a typical value.
- **Honest limitation:** the median is not perfect for the 240 zero-payment customers, for whom "0" may be more truthful. We will run a quick check in notebook 03 (fill those with 0 instead) and report whether the clusters change.

**What this cell does:** Builds the imputation as a scikit-learn Pipeline and shows that it fills blanks the same way on brand new data.  
**What to look for:** Zero missing values after transform, and the learned medians. The new row should get the same median as the training data.  
**What the output means:** A Pipeline remembers what it learned (here, the medians) when fitted. Calling transform later on new customers fills blanks using the training medians, never the new data's own, which avoids data leakage (information from outside the training data sneaking in). Notebook 02 reuses this inside the full preprocessing pipeline.

In [15]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

imputation_pipeline = Pipeline([("median_imputer", SimpleImputer(strategy="median"))]).set_output(transform="pandas")
imputed = imputation_pipeline.fit_transform(df[features])  # learn medians, then fill blanks
print(f"{imputed.isna().sum().sum()=}")
print("Learned medians for the two columns that had blanks:")
print(imputation_pipeline.named_steps["median_imputer"].statistics_[[features.index("CREDIT_LIMIT"), features.index("MINIMUM_PAYMENTS")]])

# Simulate a new customer with blanks: the pipeline fills them with the TRAINING medians
new_customer = df[features].iloc[[0]].copy()
new_customer[["CREDIT_LIMIT", "MINIMUM_PAYMENTS"]] = np.nan
imputation_pipeline.transform(new_customer)[["CREDIT_LIMIT", "MINIMUM_PAYMENTS"]]

imputed.isna().sum().sum()=np.int64(0)
Learned medians for the two columns that had blanks:
[3000.        312.343947]


,CREDIT_LIMIT,MINIMUM_PAYMENTS
0,3000.0,312.343947


### Outliers with the IQR rule

The **IQR rule** (interquartile range) flags a value as an outlier if it is more than 1.5 times the IQR below the 25th percentile (Q1) or above the 75th percentile (Q3). The IQR is Q3 minus Q1, the spread of the middle half of customers. We only *flag* outliers. We do not remove them, because a high-value customer is exactly the kind of person a segmentation should find.

**What this cell does:** Flags outliers for every feature with the IQR rule and counts them.  
**What to look for:** Which features have the largest share of outliers, and how many features each customer is flagged on.  
**What the output means:** A feature with a large share of flagged rows has a heavy tail, not a few mistakes. A customer flagged on many features is consistently extreme, which looks like a real big spender or big borrower rather than a typo.

In [16]:
def iqr_outlier_flags(frame: pd.DataFrame) -> pd.DataFrame:
    """True where a value is outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR], column by column."""
    q1 = frame.quantile(0.25)
    q3 = frame.quantile(0.75)
    iqr = q3 - q1
    return (frame < q1 - 1.5 * iqr) | (frame > q3 + 1.5 * iqr)  # broadcast comparison, one bound per column


outlier_flags = iqr_outlier_flags(df[features])
outlier_summary = pd.DataFrame({
    "outliers": outlier_flags.sum(),
    "percent": (outlier_flags.mean() * 100).round(1),
}).sort_values("outliers", ascending=False)
display(outlier_summary)

flags_per_customer = outlier_flags.sum(axis=1)  # on how many features is each customer flagged?
print(flags_per_customer.value_counts().sort_index().to_string())
print(f"{(flags_per_customer >= 5).sum()=} customers are flagged on 5 or more features")

,outliers,percent
BALANCE_FREQUENCY,1493,16.7
PRC_FULL_PAYMENT,1474,16.5
TENURE,1366,15.3
CASH_ADVANCE,1030,11.5
ONEOFF_PURCHASES,1013,11.3
INSTALLMENTS_PURCHASES,867,9.7
MINIMUM_PAYMENTS,841,9.4
PAYMENTS,808,9.0
PURCHASES,808,9.0
CASH_ADVANCE_TRX,804,9.0


0     2994
1     2549
2     1596
3      732
4      436
5      298
6      177
7      102
8       36
9       20
10       5
11       5
(flags_per_customer >= 5).sum()=np.int64(643) customers are flagged on 5 or more features


**What this cell does:** Shows the top 5 customers for purchases, cash advance and balance, together with simple ratios that would reveal data errors.  
**What to look for:** Do the numbers make sense together? For example, purchases that match the number of transactions, a high credit limit for big spenders, and payments in a similar range.  
**What the output means:** Data errors tend to look inconsistent (a huge purchase total with 0 transactions, a balance far above the credit limit for no reason). Real high-value customers look internally consistent.

In [17]:
inspection_columns = ["BALANCE", "PURCHASES", "PURCHASES_TRX", "CASH_ADVANCE", "CASH_ADVANCE_TRX", "CREDIT_LIMIT", "PAYMENTS", "TENURE"]
for column in ["PURCHASES", "CASH_ADVANCE", "BALANCE"]:
    top = df.nlargest(5, column)[inspection_columns].copy()
    top["avg_purchase_size"] = (top["PURCHASES"] / top["PURCHASES_TRX"]).round(0)   # money per purchase
    top["purchases_vs_limit"] = (top["PURCHASES"] / top["CREDIT_LIMIT"]).round(1)   # spend as a multiple of the limit
    print(f"Top 5 by {column}")
    display(top.round(0))

Top 5 by PURCHASES


,BALANCE,PURCHASES,PURCHASES_TRX,CASH_ADVANCE,CASH_ADVANCE_TRX,CREDIT_LIMIT,PAYMENTS,TENURE,avg_purchase_size,purchases_vs_limit
550,11548.0,49040.0,101,558.0,1,22500.0,46931.0,12,486.0,2.0
501,13479.0,41050.0,157,0.0,0,17000.0,36067.0,12,261.0,2.0
1256,4011.0,40041.0,358,0.0,0,18000.0,33995.0,12,112.0,2.0
1604,3392.0,38903.0,195,0.0,0,18000.0,35844.0,12,200.0,2.0
3937,2998.0,35131.0,344,0.0,0,12000.0,28897.0,12,102.0,3.0


Top 5 by CASH_ADVANCE


,BALANCE,PURCHASES,PURCHASES_TRX,CASH_ADVANCE,CASH_ADVANCE_TRX,CREDIT_LIMIT,PAYMENTS,TENURE,avg_purchase_size,purchases_vs_limit
2159,10905.0,432.0,21,47137.0,123,19600.0,39049.0,12,21.0,0.0
1059,8823.0,3719.0,61,29282.0,26,15500.0,28151.0,12,61.0,0.0
71,2990.0,4523.0,33,27296.0,27,7000.0,28233.0,12,137.0,1.0
7254,4530.0,1751.0,36,26269.0,10,8500.0,25204.0,12,49.0,0.0
7645,7081.0,0.0,0,26194.0,69,9000.0,20191.0,11,NaN,0.0


Top 5 by BALANCE


,BALANCE,PURCHASES,PURCHASES_TRX,CASH_ADVANCE,CASH_ADVANCE_TRX,CREDIT_LIMIT,PAYMENTS,TENURE,avg_purchase_size,purchases_vs_limit
138,19043.0,22010.0,216,0.0,0,18000.0,23019.0,12,102.0,1.0
4140,18496.0,5288.0,76,0.0,0,22000.0,4246.0,12,70.0,0.0
5488,16305.0,1771.0,9,7424.0,13,19000.0,5338.0,12,197.0,0.0
6629,16259.0,5025.0,143,329.0,2,18000.0,4987.0,12,35.0,0.0
5281,16116.0,685.0,15,4354.0,15,18000.0,3546.0,12,46.0,0.0


**What this cell does:** Runs simple consistency checks that would point to data errors.  
**What to look for:** Counts that are zero or tiny. Large counts mean a rule in the data we do not understand yet.  
**What the output means:** Each line is a rule that clean data should follow. Few violations means the extreme values are probably real behaviour, not mistakes.

In [18]:
frequency_columns = [column for column in features if column.endswith("_FREQUENCY") or column == "PRC_FULL_PAYMENT"]
checks = {
    "negative values anywhere": int((df[features] < 0).sum().sum()),
    "ratio columns outside 0 to 1": int(((df[frequency_columns] < 0) | (df[frequency_columns] > 1)).sum().sum()),
    "purchases > 0 but 0 purchase transactions": int(((df["PURCHASES"] > 0) & (df["PURCHASES_TRX"] == 0)).sum()),
    "cash advance > 0 but 0 cash advance transactions": int(((df["CASH_ADVANCE"] > 0) & (df["CASH_ADVANCE_TRX"] == 0)).sum()),
    "one-off purchases above total purchases": int((df["ONEOFF_PURCHASES"] > df["PURCHASES"] + 0.01).sum()),
    "balance above credit limit": int((df["BALANCE"] > df["CREDIT_LIMIT"]).sum()),
    "purchases above 5x credit limit": int((df["PURCHASES"] > 5 * df["CREDIT_LIMIT"]).sum()),
    "minimum payment above payments": int((df["MINIMUM_PAYMENTS"] > df["PAYMENTS"]).sum()),
}
pd.Series(checks, name="rows").to_frame()

,rows
negative values anywhere,0
ratio columns outside 0 to 1,8
purchases > 0 but 0 purchase transactions,3
cash advance > 0 but 0 cash advance transactions,0
one-off purchases above total purchases,3
balance above credit limit,227
purchases above 5x credit limit,9
minimum payment above payments,2365


### What the outliers tell us

- **The IQR rule flags a lot, and that is itself a finding.** Between 3% and 12% of customers are flagged on each money column, because these columns are heavy-tailed. 643 customers (7.2%) are flagged on five or more features at once. Being extreme on many features together points to genuinely different customers, not typos.
- **The bounded columns are flagged for the wrong reason.** `BALANCE_FREQUENCY` (16.7%), `PRC_FULL_PAYMENT` (16.5%) and `TENURE` (15.3%) show many "outliers" only because most customers sit at one value (1.0, 0, 12), which makes the IQR near zero. The IQR rule is not meaningful for these three, so we ignore their flags.
- **The top customers look real.**
  - *Top spenders:* purchases of 35,000 to 49,000 over 100 to 358 transactions (typical purchase about 100 to 490), credit limits of 12,000 to 22,500, and payments close to the purchase total. These are heavy users who pay what they spend.
  - *Top cash advance users:* up to 47,137 in cash over 123 transactions, with payments of 39,000. Large but internally consistent.
  - *Top balances:* about 16,000 to 19,000 on credit limits of 18,000 to 22,000. Plausible.
- **Real data errors exist but are tiny.** Out of 8,950 rows: 8 rows have `CASH_ADVANCE_FREQUENCY` above 1 (a share cannot exceed 100%), 3 have purchases but zero purchase transactions, and 3 have one-off purchases above total purchases. That is about 14 rows (0.2%), too few to matter. 227 customers (2.5%) have a balance above their credit limit and 9 spent more than 5 times their limit over 6 months while repaying a lot; both are plausible when someone repays and spends again.
- **A caution about `MINIMUM_PAYMENTS` above `PAYMENTS`.** This happens for 2,365 customers (26%). It could mean customers who pay less than the minimum due, or it could be a difference in how the two columns are measured (we have no data dictionary). We treat it as behaviour, not error, but we will not build a business claim on it.

### Decision

**Keep every row.** We do not remove outliers: the high-value customers are the segment a bank most wants to find. Instead, the log transform in Step 5 will tame their pull on the distances, and the 14 inconsistent rows are left as they are.